In [11]:
#
#Question 1. Load the dataset and inspect its dimensions, columns, and data types?

import pandas as pd
import numpy as np

df = pd.read_csv("28_tournament_match_stats.csv")

print("Dataset size:", df.shape)
print("Columns:", df.columns)

Dataset size: (609, 10)
Columns: Index(['Match_ID', 'Match_Date', 'Team', 'Opponent', 'Player_Name',
       'Player_Points', 'Match_Duration_Mins', 'Match_Type',
       'Tournament_Round', 'Result'],
      dtype='str')


In [2]:
#Question 2. Check for missing values and duplicate Match_ID records?

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate Match_ID records:", df.duplicated("Match_ID").sum())
print("Exact duplicate rows:", df.duplicated().sum())

# Each Match_ID should be counted only once.
df = df.drop_duplicates(subset="Match_ID").copy()
print("\nRows after removing duplicate Match_ID:", len(df))
print("Unique matches:", df["Match_ID"].nunique())

Missing values:
Match_ID                0
Match_Date              0
Team                    0
Opponent                0
Player_Name             0
Player_Points          15
Match_Duration_Mins    15
Match_Type              0
Tournament_Round        0
Result                  0
dtype: int64

Duplicate Match_ID records: 9
Exact duplicate rows: 9

Rows after removing duplicate Match_ID: 600
Unique matches: 600


In [3]:
#Question 3. Handle missing Player_Points or Match_Duration_Mins values using a justified method?

# Median is used because it is not greatly affected by extreme values.
print("Missing values before:")
print(df[["Player_Points", "Match_Duration_Mins"]].isnull().sum())

df["Player_Points"] = df["Player_Points"].fillna(df["Player_Points"].median())
df["Match_Duration_Mins"] = df["Match_Duration_Mins"].fillna(df["Match_Duration_Mins"].median())

print("\nMissing values after:")
print(df[["Player_Points", "Match_Duration_Mins"]].isnull().sum())

Missing values before:
Player_Points          15
Match_Duration_Mins    15
dtype: int64

Missing values after:
Player_Points          0
Match_Duration_Mins    0
dtype: int64


In [4]:
#Question 4. Convert Match_Date to datetime and extract month and weekday?

df["Match_Date"] = pd.to_datetime(df["Match_Date"])
df["Month"] = df["Match_Date"].dt.month_name()
df["Weekday"] = df["Match_Date"].dt.day_name()

print(df[["Match_Date", "Month", "Weekday"]].head())

  Match_Date      Month    Weekday
0 2026-05-26        May    Tuesday
1 2026-05-20        May  Wednesday
2 2025-06-25       June  Wednesday
3 2025-08-05     August    Tuesday
4 2026-09-16  September  Wednesday


In [5]:
#Question 5. Calculate total matches played, total points scored, and average match duration ?

print("Total matches:", df["Match_ID"].nunique())
print("Total points:", df["Player_Points"].sum())
print("Average match duration:", round(df["Match_Duration_Mins"].mean(), 2), "minutes")

Total matches: 600
Total points: 12826.0
Average match duration: 104.97 minutes


In [6]:
#Question 6. Rank teams by total points scored and number of matches won?

team_rank = df.groupby("Team").agg(
    Total_Points=("Player_Points", "sum"),
    Matches_Won=("Result", lambda x: (x == "Win").sum())
)

team_rank = team_rank.sort_values(
    ["Total_Points", "Matches_Won"], ascending=False
)
print(team_rank)

          Total_Points  Matches_Won
Team                               
Sharks          2025.0           29
Panthers        1809.0           28
Falcons         1754.0           28
Warriors        1626.0           28
Titans          1581.0           20
Lions           1372.0           20
Eagles          1332.0           25
Tigers          1327.0           25


In [7]:
#Question 7. Identify the top 10 players by total points ?

top_players = df.groupby("Player_Name")["Player_Points"].sum()
top_players = top_players.sort_values(ascending=False).head(10)
print(top_players)

Player_Name
Player 43    381.0
Player 21    359.0
Player 50    346.0
Player 60    326.0
Player 26    294.0
Player 58    280.0
Player 63    279.0
Player 46    258.0
Player 8     254.0
Player 31    250.0
Name: Player_Points, dtype: float64


In [8]:
#Question 8. Compare average points across match types or tournament rounds?

print("Average points by match type:")
print(df.groupby("Match_Type")["Player_Points"].mean().round(2))

print("\nAverage points by tournament round:")
print(df.groupby("Tournament_Round")["Player_Points"].mean().round(2))

Average points by match type:
Match_Type
Final         22.32
Friendly      21.86
League        20.97
Semi-Final    20.25
Name: Player_Points, dtype: float64

Average points by tournament round:
Tournament_Round
Final            20.42
Quarter-Final    22.25
Round 1          20.29
Round 2          22.71
Semi-Final       21.18
Name: Player_Points, dtype: float64


In [9]:
#Question 9. Calculate each team's win percentage?
team_data = df.groupby("Team").agg(
    Matches=("Match_ID", "nunique"),
    Wins=("Result", lambda x: (x == "Win").sum())
)

team_data["Win_Percentage"] = (
    team_data["Wins"] / team_data["Matches"] * 100
).round(2)

print(team_data.sort_values("Win_Percentage", ascending=False))

          Matches  Wins  Win_Percentage
Team                                   
Panthers       72    28           38.89
Tigers         65    25           38.46
Warriors       75    28           37.33
Eagles         69    25           36.23
Sharks         90    29           32.22
Falcons        90    28           31.11
Lions          66    20           30.30
Titans         73    20           27.40


In [10]:
#Question 10. Identify the longest and shortest matches?

longest = df.loc[df["Match_Duration_Mins"].idxmax()]
shortest = df.loc[df["Match_Duration_Mins"].idxmin()]

print("Longest match:")
print(longest[["Match_ID", "Match_Date", "Team", "Match_Duration_Mins"]])

print("\nShortest match:")
print(shortest[["Match_ID", "Match_Date", "Team", "Match_Duration_Mins"]])

Longest match:
Match_ID                           MA00076
Match_Date             2025-09-20 00:00:00
Team                                Sharks
Match_Duration_Mins                  179.0
Name: 76, dtype: object

Shortest match:
Match_ID                           MA00017
Match_Date             2025-05-21 00:00:00
Team                                 Lions
Match_Duration_Mins                   30.0
Name: 17, dtype: object


In [6]:
#Question 11. Use a tuple to store fixed match-result labels and a set to identify all participating teams?
 
result_labels = ("Win", "Loss", "Draw")
teams = set(df["Team"])

print("Result labels:", result_labels)
print("Participating teams:", teams)
print("Number of teams:", len(teams))

Result labels: ('Win', 'Loss', 'Draw')
Participating teams: {'Panthers', 'Tigers', 'Titans', 'Warriors', 'Falcons', 'Lions', 'Sharks', 'Eagles'}
Number of teams: 8


In [7]:
#Question 12. Use a dictionary to store team-wise wins and write a function to calculate a team's win percentag?

team_wins = df.groupby("Team")["Result"].apply(
    lambda x: (x == "Win").sum()
).to_dict()

team_matches = df.groupby("Team")["Match_ID"].nunique().to_dict()

def win_percentage(team):
    return round(team_wins[team] / team_matches[team] * 100, 2)

print("Team wins:", team_wins)
print("\nWin percentages:")
for team in team_wins:
    print(team, ":", win_percentage(team), "%")

Team wins: {'Eagles': 25, 'Falcons': 28, 'Lions': 20, 'Panthers': 28, 'Sharks': 29, 'Tigers': 25, 'Titans': 20, 'Warriors': 28}

Win percentages:
Eagles : 36.23 %
Falcons : 31.11 %
Lions : 30.3 %
Panthers : 38.89 %
Sharks : 32.22 %
Tigers : 38.46 %
Titans : 27.4 %
Warriors : 37.33 %


In [8]:
#Question 13. Use NumPy to calculate the mean, median, and standard deviation of player points?

points = df["Player_Points"].to_numpy()

print("Mean:", round(np.mean(points), 2))
print("Median:", round(np.median(points), 2))
print("Standard deviation:", round(np.std(points), 2))

Mean: 21.38
Median: 21.0
Standard deviation: 12.64


In [8]:
#Question 14. Identify unusually high player scores using IQR and create player-performance groups using documented thresholds?

import pandas as pd

# This cell is also runnable by itself after a kernel restart.
if "df" not in globals():
    df = pd.read_csv("28_tournament_match_stats.csv")
    df = df.drop_duplicates(subset="Match_ID").copy()

# Ignore missing Player_Points values when calculating thresholds.
scores = df["Player_Points"].dropna()
q1 = scores.quantile(0.25)
q3 = scores.quantile(0.75)
iqr = q3 - q1
upper_limit = q3 + 1.5 * iqr

high_scores = df[df["Player_Points"] > upper_limit]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("High score limit:", upper_limit)
print("\nUnusually high scores:")
print(high_scores[["Player_Name", "Team", "Player_Points"]])

# Low: below Q1, Medium: Q1 to Q3, High: above Q3.
def performance_group(points):
    if pd.isna(points):
        return "Unknown"
    if points < q1:
        return "Low"
    if points <= q3:
        return "Medium"
    return "High"

df["Performance_Group"] = df["Player_Points"].apply(performance_group)
print("\nPerformance group counts:")
print(df["Performance_Group"].value_counts())

Q1: 10.0
Q3: 32.0
IQR: 22.0
High score limit: 65.0

Unusually high scores:
Empty DataFrame
Columns: [Player_Name, Team, Player_Points]
Index: []

Performance group counts:
Performance_Group
Medium     308
High       146
Low        140
Unknown     15
Name: count, dtype: int64


In [9]:

# Question 15

class Player:
    def __init__(self, name, team, points):
        self.name = name
        self.team = team
        self.points = points

    def show(self):
        print("Player:", self.name)
        print("Team:", self.team)
        print("Total Points:", self.points)


# Find best player
players = df.groupby("Player_Name")["Player_Points"].sum()
name = players.idxmax()
points = players.max()
team = df[df["Player_Name"] == name]["Team"].iloc[0]

p1 = Player(name, team, points)

print("Best Player Details:")
p1.show()


# Five observations
print("\nFive Observations:")

print("1. Total matches:", df["Match_ID"].nunique())

print("2. Total players:", df["Player_Name"].nunique())

print("3. Total teams:", df["Team"].nunique())

print("4. Average player points:", round(df["Player_Points"].mean(), 2))

print("5. Average match duration:",
      round(df["Match_Duration_Mins"].mean(), 2), "minutes")


Best Player Details:
Player: Player 21
Team: Falcons
Total Points: 369.0

Five Observations:
1. Total matches: 600
2. Total players: 79
3. Total teams: 8
4. Average player points: 21.44
5. Average match duration: 105.09 minutes
